In [ ]:
# ── Cell 1: Install ───────────────────────────────────────────
!pip install flask transformers accelerate pillow pyngrok -q

In [ ]:
!pip install uvicorn==0.20.0 -q
# ── Replace Cell 1 install ────────────────────────────────────
!pip install flask transformers accelerate pillow -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.9/56.9 kB 6.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.26.0 requires uvicorn<1.0.0,>=0.34.0, but you have uvicorn 0.20.0 which is incompatible.
python-fasthtml 0.12.47 requires uvicorn[standard]>=0.30, but you have uvicorn 0.20.0 which is incompatible.
mcp 1.26.0 requires uvicorn>=0.31.1; sys_platform != "emscripten", but you have uvicorn 0.20.0 which is incompatible.


In [ ]:
import nest_asyncio
nest_asyncio.apply()

from fastapi import FastAPI, UploadFile, File
from fastapi.middleware.cors import CORSMiddleware
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor
from huggingface_hub import login
from google.colab import userdata
from PIL import Image
import uvicorn, torch, io, threading

In [ ]:
PYDEVD_DISABLE_FILE_VALIDATION=1

In [ ]:
# grab token from Colab secrets (🔑 icon on left sidebar)
# add a secret named HF_TOKEN and paste your token there

token = userdata.get("HF_TOKEN")
login(token=token)
print("✅ HuggingFace authenticated")

# ✅ Use the correct class for Qwen2-VL
model = Qwen2VLForConditionalGeneration.from_pretrained(
    "Qwen/Qwen2-VL-7B-Instruct",
    torch_dtype=torch.float16,
    device_map="auto",
    token=token
)

# ✅ Qwen2-VL uses AutoProcessor not AutoTokenizer
processor = AutoProcessor.from_pretrained(
    "Qwen/Qwen2-VL-7B-Instruct",
    token=token
)
print("✅ Model loaded!")

✅ HuggingFace authenticated


config.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/730 [00:00<?, ?it/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

✅ Model loaded!


In [ ]:
app = FastAPI()

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"]
)

@app.get("/health")
def health():
    return {"status": "ok"}

@app.post("/infer")
async def infer(
    file: UploadFile = File(...),
    prompt: str = "Describe what is happening in this image"
):
    contents = await file.read()
    image = Image.open(io.BytesIO(contents)).convert("RGB")

    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": prompt}
            ]
        }
    ]

    # ✅ use processor not tokenizer
    text = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = processor(
        text=[text],
        images=[image],
        return_tensors="pt"
    ).to("cuda")

    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=512)

    generated = processor.decode(
        output[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )

    return {"report": generated}

In [ ]:
# ── Cell 5: Run with ngrok ────────────────────────────────────
from flask import Flask, request, jsonify
from PIL import Image
from pyngrok import ngrok
import io, threading, time

flask_app = Flask(__name__)


def build_system_prompt(childName: str, context: str):
    return f"""You are an early childhood educator writing developmental observation reports.

You will be given an image of a classroom activity. Generate a structured report following this EXACT format:

---

CONTEXT:
${context}

OBSERVATION:
[Write a warm, detailed narrative in third person describing exactly what ${childName} is doing in the image.
Include specific details about materials used, actions taken, focus level, and interactions visible.
If there is a teacher-child dialogue visible or implied, write it as:
Teacher: "..."
${childName}: "..."
Write as a continuous narrative, multiple paragraphs if needed. Be descriptive and professional but warm.]

LEARNING ANALYSIS:

Language & Literacy: [Write a sentence describing how this activity supported ${childName}'s language and literacy development — e.g. sequencing skills, descriptive vocabulary, story planning, writing, or verbal expression.]

Creative Expression: [Write a sentence about ${childName}'s originality, attention to detail in designing characters, settings, props, or artistic choices.]

Cultural Awareness: [Write a sentence about connections ${childName} made between past and present, appreciation of local heritage, or cultural elements explored.]

Collaboration & Social Skills: [Write a sentence about how ${childName} worked cooperatively, listened, contributed ideas, or interacted with peers.]

Cognitive Development: [Write a sentence about problem-solving, planning skills, structuring ideas, or critical thinking ${childName} demonstrated.]

Fine Motor & Design Thinking: [Write a sentence about precision, spatial awareness, construction skills, cutting, folding, assembling, or design work ${childName} performed.]

---

RULES:
- Only include Learning Analysis categories that are clearly evidenced in the image
- Minimum 2 categories per report
- Each category description must reference ${childName} by name with a specific observable action
- Never guess or assume what is not visible in the image
- Do not identify or label any other children visible in the image
- Write observation as a continuous narrative, not bullet points
- Match the tone of a Singapore early childhood educator`;
"""


@flask_app.route("/health")
def health():
    return jsonify({"status": "ok"})


@flask_app.route("/infer", methods=["POST"])
def infer():
    try:
        file = request.files["file"]
        child_name = request.form.get("child_name", "The child")
        context = request.form.get("context", "Classroom activity")

        prompt = build_system_prompt(child_name, context)
        image = Image.open(io.BytesIO(file.read())).convert("RGB")

        messages = [{"role": "user", "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": prompt}
        ]}]

        text = processor.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        inputs = processor(
            text=[text], images=[image], return_tensors="pt"
        ).to("cuda")

        import torch
        with torch.no_grad():
            output = model.generate(**inputs, max_new_tokens=1024)

        generated = processor.decode(
            output[0][inputs["input_ids"].shape[1]:],
            skip_special_tokens=True
        )
        return jsonify({"report": generated})
    except Exception as e:
        import traceback
        traceback.print_exc()
        return jsonify({"error": str(e)}), 500


# ── start ngrok ───────────────────────────────────────────────
ngrok.set_auth_token("2rtWk9p4c162sWia1aCQ8ekGIW0_2gfN4oEyUHgXJutWoGNas")

thread = threading.Thread(
    target=lambda: flask_app.run(host="0.0.0.0", port=8000, use_reloader=False),
    daemon=True
)
thread.start()
time.sleep(2)

public_url = ngrok.connect(8000)
print(f"\n✅ API live at: {public_url}")
print(f"   Health:    {public_url}/health")
print(f"   Infer:     {public_url}/infer")


 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:8000
 * Running on http://172.28.0.12:8000
INFO:werkzeug:Press CTRL+C to quit



✅ API live at: NgrokTunnel: "https://e7b6-35-197-152-251.ngrok-free.app" -> "http://localhost:8000"
   Health:    NgrokTunnel: "https://e7b6-35-197-152-251.ngrok-free.app" -> "http://localhost:8000"/health
   Infer:     NgrokTunnel: "https://e7b6-35-197-152-251.ngrok-free.app" -> "http://localhost:8000"/infer


In [ ]:
import requests
# Test with a simple image
resp = requests.get("https://picsum.photos/200")
files = {"file": ("test.jpg", resp.content, "image/jpeg")}
result = requests.post("http://localhost:8000/infer", files=files, data={"prompt": "Describe this image"})
print(result.status_code, result.json())

INFO:werkzeug:127.0.0.1 - - [05/Mar/2026 03:01:23] "POST /infer HTTP/1.1" 200 -


200 {'report': '---\n\nCONTEXT:\nA child is engaged in a classroom activity involving the creation of a bluebonnet flower arrangement.\n\nOBSERVATION:\n$The child, a young girl with curly hair, is meticulously arranging bluebonnet flowers in a clear glass vase. She is carefully selecting the most vibrant blue flowers and placing them in the vase, ensuring each one is positioned to create a visually appealing bouquet. The child\'s focus is intense as she works, occasionally looking up to adjust the flowers or to ask for assistance from the teacher. The teacher, standing nearby, is observing the child\'s progress and providing gentle guidance. The child seems to be following a specific plan, perhaps one she has created herself, as she moves the flowers around the vase, making sure each one is in the right spot. The teacher and child are engaged in a conversation, with the teacher asking questions to help the child refine her arrangement.\n\nLEARNING ANALYSIS:\n\nLanguage & Literacy: The 